In [2]:
from google.colab import userdata

token = userdata.get("GITHUB_TOKEN")

print("Token loaded:", token is not None)


Token loaded: True


In [3]:
import os

os.environ["GITHUB_TOKEN"] = token

!git clone https://$GITHUB_TOKEN@github.com/nehnamehranmk638-dev/multilingual-rag-research.git



Cloning into 'multilingual-rag-research'...
remote: Enumerating objects: 120, done.
remote: Counting objects: 100% (120/120), done.
remote: Compressing objects: 100% (100/100), done.
remote: Total 120 (delta 53), reused 75 (delta 16), pack-reused 0 (from 0)
Receiving objects: 100% (120/120), 920.32 KiB | 2.87 MiB/s, done.
Resolving deltas: 100% (53/53), done.


In [4]:
%cd /content/multilingual-rag-research

/content/multilingual-rag-research


In [5]:
!git config --global credential.helper store

In [6]:
import subprocess

username = "nehnamehrankmk638-dev"

credential = f"""protocol=https
host=github.com
username={username}
password={token}

"""

subprocess.run(
    ["git", "credential", "approve"],
    input=credential,
    text=True,
    check=True
)

print("GitHub authentication configured.")


GitHub authentication configured.


In [7]:
!git fetch origin
!git switch nehna

branch 'nehna' set up to track 'origin/nehna'.
Switched to a new branch 'nehna'


In [8]:
!git pull

Already up to date.


In [9]:
import json
import os
import csv

In [10]:
with open("data/questions_ml.json", "r", encoding="utf-8") as f:
    questions_ml = json.load(f)

with open("results/bm25_top10_ml.json", "r", encoding="utf-8") as f:
    bm25_results_ml = json.load(f)

with open("results/dense_top10_ml.json", "r", encoding="utf-8") as f:
    dense_results_ml = json.load(f)

print("Questions:", len(questions_ml))
print("BM25 results:", len(bm25_results_ml))
print("Dense results:", len(dense_results_ml))

Questions: 100
BM25 results: 100
Dense results: 100


In [15]:
question_ids_ml = {str(q["question_id"]) for q in questions_ml}

bm25_ids_ml = {str(k) for k in bm25_results_ml.keys()}
dense_ids_ml = {str(k) for k in dense_results_ml.keys()}

print("Question IDs:", len(question_ids_ml))
print("BM25 IDs:", len(bm25_ids_ml))
print("Dense IDs:", len(dense_ids_ml))

assert question_ids_ml == bm25_ids_ml
assert question_ids_ml == dense_ids_ml

print("✓ Question IDs match across all three datasets.")

Question IDs: 100
BM25 IDs: 100
Dense IDs: 100
✓ Question IDs match across all three datasets.


In [16]:
bm25_results_ml = {
    str(k): v for k, v in bm25_results_ml.items()
}

dense_results_ml = {
    str(k): v for k, v in dense_results_ml.items()
}

In [17]:
def reciprocal_rank_fusion(list1, list2, k=60):
    scores = {}

    for rank, doc_id in enumerate(list1, start=1):
        scores[doc_id] = scores.get(doc_id, 0) + 1 / (k + rank)

    for rank, doc_id in enumerate(list2, start=1):
        scores[doc_id] = scores.get(doc_id, 0) + 1 / (k + rank)

    ranked_docs = sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    return [doc_id for doc_id, score in ranked_docs]

In [21]:
qid = str(questions_ml[0]["question_id"])

hybrid_test = reciprocal_rank_fusion(
    bm25_results_ml[qid],
    dense_results_ml[qid],
    k=60
)

print("Question ID:", qid)
print("BM25:", bm25_results_ml[qid])
print("Dense:", dense_results_ml[qid])
print("Hybrid:", hybrid_test[:10])

Question ID: 323
BM25: [49, 232, 237, 144, 55, 186, 71, 215, 62, 134]
Dense: [49, 232, 190, 231, 134, 133, 126, 85, 160, 141]
Hybrid: [49, 232, 134, 237, 190, 144, 231, 55, 186, 133]


In [20]:
bm25_results_ml = {
    str(k): v for k, v in bm25_results_ml.items()
}

dense_results_ml = {
    str(k): v for k, v in dense_results_ml.items()
}

print("✓ BM25 and Dense keys converted to strings.")

✓ BM25 and Dense keys converted to strings.


In [24]:
hybrid_results_ml = {}

for q in questions_ml:
    qid = str(q["question_id"])

    fused = reciprocal_rank_fusion(
        bm25_results_ml[qid],
        dense_results_ml[qid],
        k=60
    )

    hybrid_results_ml[qid] = fused[:10]

print("Hybrid results generated:", len(hybrid_results_ml))

Hybrid results generated: 100


In [23]:
# Normalize question IDs to strings
bm25_results_ml = {str(k): v for k, v in bm25_results_ml.items()}
dense_results_ml = {str(k): v for k, v in dense_results_ml.items()}

print("BM25 first key:", list(bm25_results_ml.keys())[0], type(list(bm25_results_ml.keys())[0]))
print("Dense first key:", list(dense_results_ml.keys())[0], type(list(dense_results_ml.keys())[0]))

BM25 first key: 323 <class 'str'>
Dense first key: 323 <class 'str'>


In [25]:
def recall_at_k(questions, results, k):
    hits = 0

    for q in questions:
        qid = str(q["question_id"])
        gold_id = q["gold_passage_id"]

        retrieved = results[qid][:k]

        if gold_id in retrieved:
            hits += 1

    return hits / len(questions)


def mrr(questions, results):
    reciprocal_ranks = []

    for q in questions:
        qid = str(q["question_id"])
        gold_id = q["gold_passage_id"]

        retrieved = results[qid]

        if gold_id in retrieved:
            rank = retrieved.index(gold_id) + 1
            reciprocal_ranks.append(1 / rank)
        else:
            reciprocal_ranks.append(0)

    return sum(reciprocal_ranks) / len(questions)

In [26]:
for k in [1, 3, 5, 10]:
    print(
        f"Malayalam Hybrid Recall@{k}:",
        recall_at_k(questions_ml, hybrid_results_ml, k)
    )

print(
    "Malayalam Hybrid MRR:",
    mrr(questions_ml, hybrid_results_ml)
)

Malayalam Hybrid Recall@1: 0.68
Malayalam Hybrid Recall@3: 0.87
Malayalam Hybrid Recall@5: 0.91
Malayalam Hybrid Recall@10: 0.97
Malayalam Hybrid MRR: 0.7831785714285714


In [27]:
print("===== MALAYALAM RETRIEVAL COMPARISON =====")

for k in [1, 3, 5, 10]:
    bm25_score = recall_at_k(questions_ml, bm25_results_ml, k)
    dense_score = recall_at_k(questions_ml, dense_results_ml, k)
    hybrid_score = recall_at_k(questions_ml, hybrid_results_ml, k)

    print(f"\nRecall@{k}")
    print(f"BM25   : {bm25_score:.4f}")
    print(f"Dense  : {dense_score:.4f}")
    print(f"Hybrid : {hybrid_score:.4f}")

print("\nMRR")
print(f"BM25   : {mrr(questions_ml, bm25_results_ml):.4f}")
print(f"Dense  : {mrr(questions_ml, dense_results_ml):.4f}")
print(f"Hybrid : {mrr(questions_ml, hybrid_results_ml):.4f}")

===== MALAYALAM RETRIEVAL COMPARISON =====

Recall@1
BM25   : 0.6500
Dense  : 0.6600
Hybrid : 0.6800

Recall@3
BM25   : 0.7500
Dense  : 0.8400
Hybrid : 0.8700

Recall@5
BM25   : 0.8200
Dense  : 0.9000
Hybrid : 0.9100

Recall@10
BM25   : 0.8900
Dense  : 0.9300
Hybrid : 0.9700

MRR
BM25   : 0.7152
Dense  : 0.7567
Hybrid : 0.7832


In [28]:
hybrid_rescues_ml = []

for q in questions_ml:
    qid = str(q["question_id"])
    gold_id = q["gold_passage_id"]

    bm25_top5 = bm25_results_ml[qid][:5]
    dense_top5 = dense_results_ml[qid][:5]
    hybrid_top5 = hybrid_results_ml[qid][:5]

    if (
        gold_id not in bm25_top5
        and gold_id not in dense_top5
        and gold_id in hybrid_top5
    ):
        hybrid_rescues_ml.append(q)

print("Number of hybrid rescues at k=5:", len(hybrid_rescues_ml))

Number of hybrid rescues at k=5: 0


In [30]:
import json
import os

os.makedirs("results", exist_ok=True)

with open("results/hybrid_top10_ml.json", "w", encoding="utf-8") as f:
    json.dump(
        hybrid_results_ml,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Saved: results/hybrid_top10_ml.json")

Saved: results/hybrid_top10_ml.json


In [31]:
import csv

with open(
    "results/retrieval_comparison_ml.csv",
    "w",
    newline="",
    encoding="utf-8"
) as f:

    writer = csv.writer(f)

    writer.writerow([
        "k",
        "bm25_recall",
        "dense_recall",
        "hybrid_recall"
    ])

    for k in [1, 3, 5, 10]:
        writer.writerow([
            k,
            recall_at_k(questions_ml, bm25_results_ml, k),
            recall_at_k(questions_ml, dense_results_ml, k),
            recall_at_k(questions_ml, hybrid_results_ml, k)
        ])

print("Saved: results/retrieval_comparison_ml.csv")

Saved: results/retrieval_comparison_ml.csv


In [32]:
import os

print(
    "Hybrid results:",
    os.path.exists("results/hybrid_top10_ml.json")
)

print(
    "Comparison CSV:",
    os.path.exists("results/retrieval_comparison_ml.csv")
)

Hybrid results: True
Comparison CSV: True


In [33]:
print("========== MALAYALAM PHASE 1 SUMMARY ==========")

print("\nBM25")
for k in [1, 3, 5, 10]:
    print(f"Recall@{k}: {recall_at_k(questions_ml, bm25_results_ml, k):.4f}")
print(f"MRR: {mrr(questions_ml, bm25_results_ml):.4f}")

print("\nDense")
for k in [1, 3, 5, 10]:
    print(f"Recall@{k}: {recall_at_k(questions_ml, dense_results_ml, k):.4f}")
print(f"MRR: {mrr(questions_ml, dense_results_ml):.4f}")

print("\nHybrid")
for k in [1, 3, 5, 10]:
    print(f"Recall@{k}: {recall_at_k(questions_ml, hybrid_results_ml, k):.4f}")
print(f"MRR: {mrr(questions_ml, hybrid_results_ml):.4f}")

print("\nHybrid Top-5 rescues:", len(hybrid_rescues_ml))

========== MALAYALAM PHASE 1 SUMMARY ==========

BM25
Recall@1: 0.6500
Recall@3: 0.7500
Recall@5: 0.8200
Recall@10: 0.8900
MRR: 0.7152

Dense
Recall@1: 0.6600
Recall@3: 0.8400
Recall@5: 0.9000
Recall@10: 0.9300
MRR: 0.7567

Hybrid
Recall@1: 0.6800
Recall@3: 0.8700
Recall@5: 0.9100
Recall@10: 0.9700
MRR: 0.7832

Hybrid Top-5 rescues: 0


In [34]:
!git status

On branch nehna
Your branch is up to date with 'origin/nehna'.

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	notebooks/ml_04_hybrid.ipynb
	results/hybrid_top10_ml.json
	results/retrieval_comparison_ml.csv

nothing added to commit but untracked files present (use "git add" to track)


In [36]:
!git add .

In [39]:
!git commit -m "Add Malayalam hybrid retrieval"

[nehna 91352ce] Add Malayalam hybrid retrieval
 3 files changed, 2034 insertions(+)
 create mode 100644 notebooks/ml_04_hybrid.ipynb
 create mode 100644 results/hybrid_top10_ml.json
 create mode 100644 results/retrieval_comparison_ml.csv


In [40]:
!git push

Enumerating objects: 10, done.
Counting objects: 100% (10/10), done.
Delta compression using up to 2 threads
Compressing objects: 100% (7/7), done.
Writing objects: 100% (7/7), 6.77 KiB | 3.38 MiB/s, done.
Total 7 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://github.com/nehnamehranmk638-dev/multilingual-rag-research.git
   d65545b..91352ce  nehna -> nehna


In [41]:
print("BM25 MRR:", mrr(questions_ml, bm25_results_ml))
print("Dense MRR:", mrr(questions_ml, dense_results_ml))
print("Hybrid MRR:", mrr(questions_ml, hybrid_results_ml))

BM25 MRR: 0.7151904761904762
Dense MRR: 0.7566666666666667
Hybrid MRR: 0.7831785714285714


In [42]:
print("========== MALAYALAM RETRIEVAL RESULTS ==========")

print("\nBM25")
print("Recall@1 :", recall_at_k(questions_ml, bm25_results_ml, 1))
print("Recall@3 :", recall_at_k(questions_ml, bm25_results_ml, 3))
print("Recall@5 :", recall_at_k(questions_ml, bm25_results_ml, 5))
print("Recall@10:", recall_at_k(questions_ml, bm25_results_ml, 10))
print("MRR      :", mrr(questions_ml, bm25_results_ml))

print("\nDense")
print("Recall@1 :", recall_at_k(questions_ml, dense_results_ml, 1))
print("Recall@3 :", recall_at_k(questions_ml, dense_results_ml, 3))
print("Recall@5 :", recall_at_k(questions_ml, dense_results_ml, 5))
print("Recall@10:", recall_at_k(questions_ml, dense_results_ml, 10))
print("MRR      :", mrr(questions_ml, dense_results_ml))

print("\nHybrid")
print("Recall@1 :", recall_at_k(questions_ml, hybrid_results_ml, 1))
print("Recall@3 :", recall_at_k(questions_ml, hybrid_results_ml, 3))
print("Recall@5 :", recall_at_k(questions_ml, hybrid_results_ml, 5))
print("Recall@10:", recall_at_k(questions_ml, hybrid_results_ml, 10))
print("MRR      :", mrr(questions_ml, hybrid_results_ml))

========== MALAYALAM RETRIEVAL RESULTS ==========

BM25
Recall@1 : 0.65
Recall@3 : 0.75
Recall@5 : 0.82
Recall@10: 0.89
MRR      : 0.7151904761904762

Dense
Recall@1 : 0.66
Recall@3 : 0.84
Recall@5 : 0.9
Recall@10: 0.93
MRR      : 0.7566666666666667

Hybrid
Recall@1 : 0.68
Recall@3 : 0.87
Recall@5 : 0.91
Recall@10: 0.97
MRR      : 0.7831785714285714


In [38]:
!git config --global user.email "nehnamehranmk17@gmail.com"
!git config --global user.name "nehnamehranmk638-dev"